# هیستوگرام‌های تصویر / Image Histograms

**هدف:** یادگیری محاسبه و تحلیل هیستوگرام‌های تصویر

---

## محتوا:
1. هیستوگرام چیست؟
2. محاسبه هیستوگرام یک‌بعدی
3. هیستوگرام‌های رنگی
4. هموارسازی هیستوگرام

In [ ]:
# Import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple

# تنظیمات نمایش
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. هیستوگرام چیست؟ / What is a Histogram?

**تعریف:** هیستوگرام نمایش گرافیکی توزیع شدت پیکسل‌ها در یک تصویر است.

**فرمول:**
```
h(i) = تعداد پیکسل‌هایی که مقدار آنها برابر i است
```

**کاربردها:**
- تحلیل کنتراست تصویر
- آستانه‌گذاری خودکار
- بهبود کیفیت تصویر
- مقایسه تصاویر

In [ ]:
def display_image_with_histogram(image: np.ndarray, title: str = 'Image'):
    """
    نمایش تصویر به همراه هیستوگرام آن
    
    Args:
        image: تصویر ورودی (grayscale)
        title: عنوان نمایش
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # نمایش تصویر
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title(f'{title}', fontweight='bold')
    axes[0].axis('off')
    
    # محاسبه و نمایش هیستوگرام
    hist = cv2.calcHist([image], [0], None, [256], [0, 256])
    axes[1].plot(hist, color='black')
    axes[1].set_title('Histogram', fontweight='bold')
    axes[1].set_xlabel('Pixel Intensity')
    axes[1].set_ylabel('Frequency')
    axes[1].set_xlim([0, 256])
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print("✓ توابع کمکی آماده شدند")

## 2. ایجاد تصاویر نمونه / Create Sample Images

In [ ]:
# ایجاد تصاویر با توزیع‌های مختلف

# تصویر تاریک (Low intensity)
dark_image = np.random.randint(0, 100, (256, 256), dtype=np.uint8)

# تصویر روشن (High intensity)
bright_image = np.random.randint(150, 256, (256, 256), dtype=np.uint8)

# تصویر با کنتراست کم (Low contrast)
low_contrast = np.random.randint(100, 150, (256, 256), dtype=np.uint8)

# تصویر با کنتراست بالا (High contrast)
high_contrast = np.zeros((256, 256), dtype=np.uint8)
high_contrast[:128, :] = 50
high_contrast[128:, :] = 200

print("✓ تصاویر نمونه ایجاد شدند")

In [ ]:
# نمایش تصاویر و هیستوگرام‌ها
display_image_with_histogram(dark_image, 'Dark Image')
display_image_with_histogram(bright_image, 'Bright Image')
display_image_with_histogram(low_contrast, 'Low Contrast')
display_image_with_histogram(high_contrast, 'High Contrast')

## 3. محاسبه هیستوگرام با OpenCV

**تابع cv2.calcHist():**
```python
hist = cv2.calcHist(images, channels, mask, histSize, ranges)
```

**پارامترها:**
- `images`: لیست تصاویر ورودی
- `channels`: کانال‌های مورد نظر [0] برای grayscale
- `mask`: ماسک (None برای تمام تصویر)
- `histSize`: تعداد bins (معمولاً [256])
- `ranges`: محدوده مقادیر [0, 256]

In [ ]:
# مثال: محاسبه هیستوگرام دستی
def calculate_histogram_manual(image: np.ndarray) -> np.ndarray:
    """
    محاسبه هیستوگرام به صورت دستی
    """
    hist = np.zeros(256, dtype=int)
    
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            intensity = image[i, j]
            hist[intensity] += 1
    
    return hist

# مقایسه با OpenCV
test_image = np.random.randint(0, 256, (100, 100), dtype=np.uint8)

hist_manual = calculate_histogram_manual(test_image)
hist_opencv = cv2.calcHist([test_image], [0], None, [256], [0, 256]).flatten()

print(f"تفاوت بین دو روش: {np.sum(np.abs(hist_manual - hist_opencv))}")
print("✓ هر دو روش نتیجه یکسانی دارند" if np.allclose(hist_manual, hist_opencv) else "✗ تفاوت وجود دارد")

## 4. هیستوگرام‌های رنگی / Color Histograms

برای تصاویر رنگی، هیستوگرام هر کانال (R, G, B) را جداگانه محاسبه می‌کنیم.

In [ ]:
def display_color_histogram(image_bgr: np.ndarray, title: str = 'Color Image'):
    """
    نمایش تصویر رنگی با هیستوگرام‌های RGB
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # تبدیل BGR به RGB برای نمایش
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    
    # نمایش تصویر
    axes[0].imshow(image_rgb)
    axes[0].set_title(title, fontweight='bold')
    axes[0].axis('off')
    
    # محاسبه و نمایش هیستوگرام‌ها
    colors = ('r', 'g', 'b')
    for i, color in enumerate(colors):
        hist = cv2.calcHist([image_bgr], [i], None, [256], [0, 256])
        axes[1].plot(hist, color=color, label=color.upper())
    
    axes[1].set_title('RGB Histograms', fontweight='bold')
    axes[1].set_xlabel('Pixel Intensity')
    axes[1].set_ylabel('Frequency')
    axes[1].set_xlim([0, 256])
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ایجاد تصویر رنگی نمونه
color_image = np.zeros((256, 256, 3), dtype=np.uint8)
color_image[:, :85, 2] = 200  # Red region
color_image[:, 85:170, 1] = 200  # Green region
color_image[:, 170:, 0] = 200  # Blue region

display_color_histogram(color_image, 'RGB Regions')

## 5. هموارسازی هیستوگرام / Histogram Smoothing

هیستوگرام‌ها می‌توانند نویزی باشند. برای هموارسازی از فیلتر میانگین استفاده می‌کنیم.

In [ ]:
def smooth_histogram(hist: np.ndarray, kernel_size: int = 5) -> np.ndarray:
    """
    هموارسازی هیستوگرام با فیلتر میانگین
    """
    kernel = np.ones(kernel_size) / kernel_size
    smoothed = np.convolve(hist.flatten(), kernel, mode='same')
    return smoothed

# مثال
noisy_image = np.random.randint(0, 256, (256, 256), dtype=np.uint8)
hist_original = cv2.calcHist([noisy_image], [0], None, [256], [0, 256]).flatten()
hist_smoothed = smooth_histogram(hist_original, kernel_size=11)

# نمایش
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist_original, color='blue', alpha=0.7, label='Original')
axes[0].set_title('Original Histogram', fontweight='bold')
axes[0].set_xlabel('Intensity')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

axes[1].plot(hist_smoothed, color='red', label='Smoothed')
axes[1].set_title('Smoothed Histogram (kernel=11)', fontweight='bold')
axes[1].set_xlabel('Intensity')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. تمرین عملی / Practical Exercise

**وظیفه:** 
1. یک تصویر واقعی بارگذاری کنید
2. هیستوگرام آن را محاسبه و نمایش دهید
3. تصویر را به سه ناحیه تقسیم کنید و هیستوگرام هر ناحیه را مقایسه کنید

In [ ]:
# کد تمرین شما اینجا
# TODO: بارگذاری تصویر
# TODO: محاسبه هیستوگرام
# TODO: تقسیم‌بندی و مقایسه

pass

## نتیجه‌گیری / Conclusion

**نکات کلیدی:**

1. **هیستوگرام** = توزیع شدت پیکسل‌ها
2. **کاربردها:** تحلیل کنتراست، آستانه‌گذاری، بهبود تصویر
3. **تصاویر رنگی:** هیستوگرام جداگانه برای هر کانال
4. **هموارسازی:** کاهش نویز در هیستوگرام

---

**بعدی:** `02_histogram_equalisation.ipynb` - بهبود کنتراست با یکسان‌سازی هیستوگرام